# Project D — Consolidated depth-C rerun (Tasks 1a and 1b)

This notebook runs only the **new depth-tuned** configuration (no baseline retraining):

- **depthC**: `LinearSVC` with depth-specific `C`:
  - depth 0: `C=0.1`
  - depth 1: `C=10`
  - depth 2: `C=10`
  - depth 3: `C=10`
  - fallback for other depths: `C=1.0`

It compares depth-C against **saved baseline values** from earlier notebooks/CSVs and shows results as **tables in this notebook** (no CSV export).

It reports both assignment tasks:
- **Task 1a**: H1 metrics (`h1_*`) via `train_and_summarize`.
- **Task 1b**: full-tree metrics (`ft_*`, `leaf_*`) via `train_full_tree_and_summarize`.


In [ ]:
from __future__ import annotations

import time
from pathlib import Path
import pandas as pd
from IPython.display import display

from topic_hierarchy import load_topic_tree
from hierarchical_training_data import make_multilabel_binary_pool_data
from hierarchical_classifier import (
    svm_linear_binary_edge_factory,
    linear_svc_estimator_by_depth,
    binary_edge_factory,
)
from hierarchical_summary_metrics import train_and_summarize, train_full_tree_and_summarize


def homework4_base() -> Path:
    here = Path.cwd().resolve()
    if (here / 'topics.csv').exists():
        return here
    for p in [here, *here.parents]:
        if (p / 'topics.csv').exists():
            return p
    raise FileNotFoundError('topics.csv not found — set cwd to Homework 4')


BASE = homework4_base()
tree = load_topic_tree(str(BASE / 'topics.csv'))
mldata = make_multilabel_binary_pool_data(base_path=str(BASE))

SPLIT = 'test'
RESTRICT = True
MAX_FEATURES = 8000
VERBOSE_1B = False

DEPTH_C_MAP = {0: 0.1, 1: 10.0, 2: 10.0, 3: 10.0}

print('BASE', BASE)
print('train', len(mldata.train_ids()), 'test', len(mldata.test_ids()))
print('depth-C map', DEPTH_C_MAP)


### Build depth-C factory


In [ ]:
BASELINE_1A = {
    'h1_macro_f1': 0.8671,
    'h1_macro_precision': 0.8878,
    'h1_macro_recall': 0.8483,
    'h1_pooled_micro_f1': 0.8802,
    'h1_pooled_micro_precision': 0.8938,
    'h1_pooled_micro_recall': 0.8671,
    'h1_pooled_micro_accuracy': 0.9215,
    'h1_pos_weighted_f1': 0.8798,
    'h1_pos_weighted_precision': 0.8935,
    'h1_pos_weighted_recall': 0.8671,
}

baseline_1b_df = pd.read_csv(BASE / 'problem3_phase1_baseline_tfidf_from_1b.csv')
_baseline_1b_row = baseline_1b_df[baseline_1b_df['model'] == 'SVM_tfidf'].iloc[0]
BASELINE_1B = {
    'ft_macro_f1': float(_baseline_1b_row['ft_macro_f1']),
    'ft_macro_precision': float(_baseline_1b_row['ft_macro_precision']),
    'ft_macro_recall': float(_baseline_1b_row['ft_macro_recall']),
    'ft_pooled_micro_f1': float(_baseline_1b_row['ft_pooled_micro_f1']),
    'ft_pooled_micro_precision': float(_baseline_1b_row['ft_pooled_micro_precision']),
    'ft_pooled_micro_recall': float(_baseline_1b_row['ft_pooled_micro_recall']),
    'ft_pooled_micro_accuracy': float(_baseline_1b_row['ft_pooled_micro_accuracy']),
    'ft_pos_weighted_f1': float(_baseline_1b_row['ft_pos_weighted_f1']),
    'ft_pos_weighted_precision': float(_baseline_1b_row['ft_pos_weighted_precision']),
    'ft_pos_weighted_recall': float(_baseline_1b_row['ft_pos_weighted_recall']),
    'leaf_micro_f1': float(_baseline_1b_row['leaf_micro_f1']),
    'leaf_macro_f1': float(_baseline_1b_row['leaf_macro_f1']),
    'leaf_weighted_f1': float(_baseline_1b_row['leaf_weighted_f1']),
    'leaf_samples_f1': float(_baseline_1b_row['leaf_samples_f1']),
    'path_gold_branch_recall': float(_baseline_1b_row['path_gold_branch_recall']),
}


def make_depthc_factory():
    est = linear_svc_estimator_by_depth(
        DEPTH_C_MAP,
        default_c=1.0,
        dual=False,
        max_iter=8000,
    )
    return binary_edge_factory(
        tfidf_kw=dict(min_df=2, max_features=MAX_FEATURES),
        estimator=est,
        text_vectorizer='tfidf',
    )


### Task 1a rerun (depth-C only, with full metric table)


In [ ]:
print("\n" + "=" * 72)
print("TASK 1A [depthC]")
print("=" * 72)

t0 = time.perf_counter()
_, row_1a = train_and_summarize(
    'depthC',
    tree,
    mldata,
    make_depthc_factory(),
    split=SPLIT,
    include_leaf=False,
    restrict_to_parent_subtree=RESTRICT,
)
wall_1a = time.perf_counter() - t0

depth_1a = {
    'task': '1a',
    'config': 'depthC',
    'h1_macro_f1': row_1a.get('h1_macro_f1'),
    'h1_macro_precision': row_1a.get('h1_macro_precision'),
    'h1_macro_recall': row_1a.get('h1_macro_recall'),
    'h1_pooled_micro_f1': row_1a.get('h1_pooled_micro_f1'),
    'h1_pooled_micro_precision': row_1a.get('h1_pooled_micro_precision'),
    'h1_pooled_micro_recall': row_1a.get('h1_pooled_micro_recall'),
    'h1_pooled_micro_accuracy': row_1a.get('h1_pooled_micro_accuracy'),
    'h1_pos_weighted_f1': row_1a.get('h1_pos_weighted_f1'),
    'h1_pos_weighted_precision': row_1a.get('h1_pos_weighted_precision'),
    'h1_pos_weighted_recall': row_1a.get('h1_pos_weighted_recall'),
    'wall_time_sec': wall_1a,
}

baseline_1a = {'task': '1a', 'config': 'baseline_saved', 'wall_time_sec': float('nan'), **BASELINE_1A}

df_1a = pd.DataFrame([baseline_1a, depth_1a])
for c in [
    'h1_macro_f1',
    'h1_macro_precision',
    'h1_macro_recall',
    'h1_pooled_micro_f1',
    'h1_pooled_micro_precision',
    'h1_pooled_micro_recall',
    'h1_pooled_micro_accuracy',
    'h1_pos_weighted_f1',
    'h1_pos_weighted_precision',
    'h1_pos_weighted_recall',
]:
    base = float(df_1a.loc[df_1a['config'] == 'baseline_saved', c].iloc[0])
    df_1a[f'delta_vs_baseline_{c}'] = df_1a[c] - base

print(
    f"ok h1_macro_f1={depth_1a['h1_macro_f1']:.4f} "
    f"h1_pooled_micro_f1={depth_1a['h1_pooled_micro_f1']:.4f} "
    f"h1_pooled_micro_accuracy={depth_1a['h1_pooled_micro_accuracy']:.4f} "
    f"({wall_1a:.1f}s)"
)
display(df_1a.round(4))


### Task 1b rerun (full tree, depth-C only, with full metric table)

This is the slow part.


In [ ]:
print("\n" + "=" * 72)
print("TASK 1B [depthC]")
print("=" * 72)

t0 = time.perf_counter()
_, row_1b = train_full_tree_and_summarize(
    'depthC',
    tree,
    mldata,
    make_depthc_factory(),
    split=SPLIT,
    verbose=VERBOSE_1B,
    restrict_to_parent_subtree=RESTRICT,
)
wall_1b = time.perf_counter() - t0

depth_1b = {
    'task': '1b',
    'config': 'depthC',
    'ft_macro_f1': row_1b.get('ft_macro_f1'),
    'ft_macro_precision': row_1b.get('ft_macro_precision'),
    'ft_macro_recall': row_1b.get('ft_macro_recall'),
    'ft_pooled_micro_f1': row_1b.get('ft_pooled_micro_f1'),
    'ft_pooled_micro_precision': row_1b.get('ft_pooled_micro_precision'),
    'ft_pooled_micro_recall': row_1b.get('ft_pooled_micro_recall'),
    'ft_pooled_micro_accuracy': row_1b.get('ft_pooled_micro_accuracy'),
    'ft_pos_weighted_f1': row_1b.get('ft_pos_weighted_f1'),
    'ft_pos_weighted_precision': row_1b.get('ft_pos_weighted_precision'),
    'ft_pos_weighted_recall': row_1b.get('ft_pos_weighted_recall'),
    'leaf_micro_f1': row_1b.get('leaf_micro_f1'),
    'leaf_macro_f1': row_1b.get('leaf_macro_f1'),
    'leaf_weighted_f1': row_1b.get('leaf_weighted_f1'),
    'leaf_samples_f1': row_1b.get('leaf_samples_f1'),
    'path_gold_branch_recall': row_1b.get('path_gold_branch_recall'),
    'wall_time_sec': wall_1b,
}

baseline_1b = {'task': '1b', 'config': 'baseline_saved', 'wall_time_sec': float('nan'), **BASELINE_1B}

df_1b = pd.DataFrame([baseline_1b, depth_1b])
for c in [
    'ft_macro_f1',
    'ft_macro_precision',
    'ft_macro_recall',
    'ft_pooled_micro_f1',
    'ft_pooled_micro_precision',
    'ft_pooled_micro_recall',
    'ft_pooled_micro_accuracy',
    'ft_pos_weighted_f1',
    'ft_pos_weighted_precision',
    'ft_pos_weighted_recall',
    'leaf_micro_f1',
    'leaf_macro_f1',
    'leaf_weighted_f1',
    'leaf_samples_f1',
    'path_gold_branch_recall',
]:
    base = float(df_1b.loc[df_1b['config'] == 'baseline_saved', c].iloc[0])
    df_1b[f'delta_vs_baseline_{c}'] = df_1b[c] - base

print(
    f"ok ft_pooled_micro_f1={depth_1b['ft_pooled_micro_f1']:.4f} "
    f"ft_pooled_micro_accuracy={depth_1b['ft_pooled_micro_accuracy']:.4f} "
    f"leaf_micro_f1={depth_1b['leaf_micro_f1']:.4f} ({wall_1b:.1f}s)"
)
display(df_1b.round(4))


### Consolidated output


In [ ]:
final_rows = []
if 'df_1a' in globals() and not df_1a.empty:
    final_rows.append(df_1a)
if 'df_1b' in globals() and not df_1b.empty:
    final_rows.append(df_1b)

consolidated = pd.concat(final_rows, ignore_index=True) if final_rows else pd.DataFrame()
if not consolidated.empty:
    display(consolidated.round(4))
else:
    print('No results to display')
